# CH 01 Probability


### Sample Spaces and Measures

Financial Context

Think of $\Omega$ as all possible daily returns for a stock. Because asset returns are continuous, we can't just count outcomes; we use intervals to define events (e.g., "What is the probability that Apple drops more than 5% tomorrow?").

Python Implementation: Vectorized Simulations

In quantitative finance, we rarely map out analytic sample spaces explicitly. Instead, we generate empirical sample spaces using vectorized operations in numpy to model asset behavior (e.g., Geometric Brownian Motion).


In [1]:
# In quantitative finance, we rarely map out analytic sample spaces explicitly. Instead, we generate empirical sample spaces using vectorized operations in numpy to model asset behavior (e.g., Geometric Brownian Motion).

import numpy as np
import pandas as pd

# Parameters for a stock simulation
np.random.seed(42)
n_simulations = 100_000
initial_price = 100
mu = 0.05  # Annual expected return
sigma = 0.20  # Annual volatility
dt = 1 / 252  # One trading day

# Generate the sample space of daily returns using normal distribution
# R ~ N((mu - 0.5*sigma^2)*dt, sigma*sqrt(dt))
drift = (mu - 0.5 * sigma**2) * dt
shock = sigma * np.sqrt(dt) * np.random.standard_normal(n_simulations)
daily_returns = np.exp(drift + shock) - 1

# Convert to a pandas Series for industry-standard data manipulation
returns_df = pd.Series(daily_returns, name="Daily_Returns")

### 2. Axioms and Interpretation

Wasserman highlights the axioms of probability (Kolmogorov’s axioms), but the interpretation of probability splits the finance industry into two main camps:

1. Frequentist (Relative Frequency): "If we run this trading strategy 10,000 times in backtests, it wins 55% of the time."

2. Bayesian (Subjective/Degree of Belief): "Based on current macro data, there is a 70% chance the Fed cuts rates next week." (We update this belief as new data arrives).


In [2]:
# Event A: Daily loss exceeds 2% (Return <= -0.02)
event_condition = returns_df <= -0.02

# P(A) = count(A) / total outcomes
prob_loss_exceeds_2pct = event_condition.mean()

print(f"Empirical Probability of a >2% daily loss: {prob_loss_exceeds_2pct:.4f}")

# Industry Standard: Finding the 95% Historical Value at Risk (VaR)
var_95 = returns_df.quantile(0.05)
print(f"95% Daily VaR (Empirical): {var_95:.4f}")

Empirical Probability of a >2% daily loss: 0.0529
95% Daily VaR (Empirical): -0.0204


### 3. Conditional Probability and Independence

Wasserman presents the standard definition:

$$P(A | B) = \frac{P(A \cap B)}{P(B)}$$

If $A$ and $B$ are independent, $P(A \cap B) = P(A)P(B)$.

Financial Context: Regime Switching and Market Crashes

Independence is a dangerous assumption in finance. Asset returns usually exhibit volatility clustering and asymmetric dependence (when Market Index X crashes, Stock Y is highly likely to crash with it, even if they seem independent during bull markets).


In [3]:
# Conditional probability using a joint distribution of a Market Index and a Hedge Fund.

# Simulating Joint Probabilities
# Suppose Event B = Market crashes (Return < -3%)
# Suppose Event A = Hedge Fund loses money (Return < 0%)

n_days = 10_000
market_returns = np.random.normal(0, 0.01, n_days)
# Hedge fund has some market exposure + idiosyncratic risk
fund_returns = 0.5 * market_returns + np.random.normal(0, 0.005, n_days)

df = pd.DataFrame({"Market": market_returns, "Fund": fund_returns})

# Define Events
event_B = df["Market"] <= -0.03
event_A = df["Fund"] < 0

# P(B)
prob_B = event_B.mean()
# P(A and B)
prob_A_and_B = (event_A & event_B).mean()

# Conditional Probability: P(A | B)
prob_A_given_B = prob_A_and_B / prob_B if prob_B > 0 else 0

print(f"P(Market Crashes): {prob_B:.4f}")
print(f"P(Fund Loses Money | Market Crashes): {prob_A_given_B:.4f}")

P(Market Crashes): 0.0011
P(Fund Loses Money | Market Crashes): 1.0000


### 4. Bayes' Theorem

The cornerstone of quantitative regimes, signal processing, and statistical arbitrage:

$$P(A | B) = \frac{P(B | A)P(A)}{P(B)}$$

Financial Application: Default Risk given a Credit Rating Downgrade

1. $A$: Company defaults. $P(A)$ is the prior probability of default.

2. $B$: Company gets a credit rating downgrade.


In [4]:
# Prior probability of default for a high-yield bond portfolio
prob_default = 0.03

# Probability of a downgrade given they will default (Sensitivity of rating agency)
prob_downgrade_given_default = 0.85

# Probability of a downgrade given they will NOT default (False positive rate)
prob_downgrade_given_no_default = 0.10

# Total probability of a downgrade P(B) using the Law of Total Probability
prob_no_default = 1 - prob_default
prob_downgrade = (prob_downgrade_given_default * prob_default) + (
    prob_downgrade_given_no_default * prob_no_default
)

# Bayes' Theorem: P(Default | Downgrade)
prob_default_given_downgrade = (
    prob_downgrade_given_default * prob_default
) / prob_downgrade

print(
    f"Updated Probability of Default after Downgrade: {prob_default_given_downgrade:.4f}"
)

Updated Probability of Default after Downgrade: 0.2082


### Production Standards Check

When doing this work professionally:

Use pandas and numpy vectorized operations, NOT Python loops, for speed.

Fix random seeds (np.random.seed) to ensure backtests and simulations are reproducible.

Watch out for tail risk: Standard normal distributions (used above for simplicity) underestimate financial crashes. As you advance through Wasserman, we'll swap these out for Student-t distributions or extreme value theory.
